In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

In [3]:
df = pd.read_parquet("../data/processed/spy_processed.parquet")
print(df.shape)
print(df.dtypes)
df.head(3)

(15652, 23)
symbol                                               str
expiration                                datetime64[ns]
strike                                           float64
right                                                str
timestamp               datetime64[us, America/New_York]
bid                                              float64
ask                                              float64
delta                                            float64
theta                                            float64
vega                                             float64
rho                                              float64
epsilon                                          float64
lambda                                           float64
implied_vol                                      float64
iv_error                                         float64
underlying_timestamp    datetime64[us, America/New_York]
underlying_price                                 float64
spy_close          

,symbol,expiration,strike,right,timestamp,bid,ask,delta,theta,vega,...,implied_vol,iv_error,underlying_timestamp,underlying_price,spy_close,spy_open,log_return_from_open,ttm_min,log_return,open_interest
0,SPY,2026-05-06,714.0,CALL,2026-05-06 10:30:00-04:00,16.04,16.11,0.9898,-0.4813,0.4926,...,0.3828,0.0000,2026-05-06 10:30:00-04:00,730.03,733.77,728.19,0.002524,330.0,0.005110,1060
1,SPY,2026-05-06,714.0,PUT,2026-05-06 09:45:00-04:00,0.04,0.05,-0.0158,-0.5438,0.7730,...,0.3671,0.0142,2026-05-06 09:45:00-04:00,729.15,733.77,728.19,0.001317,375.0,0.006316,4063
2,SPY,2026-05-06,714.0,PUT,2026-05-06 10:00:00-04:00,0.04,0.05,-0.0153,-0.5652,0.7359,...,0.3847,0.0027,2026-05-06 10:00:00-04:00,729.66,733.77,728.19,0.002017,360.0,0.005617,4063


## Method 1: Delta finite-difference approximation

Gamma is the rate of change of delta w.r.t. the underlying price:

$$\Gamma \approx \frac{\Delta_{t} - \Delta_{t-1}}{S_t - S_{t-1}}$$

For each contract `(expiration, strike, right)` we sort by timestamp and take consecutive differences.
Rows where the underlying barely moved (`|ΔS| < 0.01`) are dropped to avoid division-by-near-zero noise.

In [4]:
contract_keys = ["expiration", "strike", "right"]

df_sorted = df.sort_values(contract_keys + ["timestamp"]).copy()

grp = df_sorted.groupby(contract_keys, sort=False)
df_sorted["d_delta"] = grp["delta"].diff()
df_sorted["d_S"] = grp["underlying_price"].diff()

# Drop first timestamp per contract (no prior row) and near-zero price moves
mask = df_sorted["d_delta"].notna() & (df_sorted["d_S"].abs() >= 0.01)
df_fd = df_sorted[mask].copy()
df_fd["gamma_fd"] = df_fd["d_delta"] / df_fd["d_S"]

print(f"Rows with finite-difference gamma: {len(df_fd):,}")
df_fd[["expiration", "strike", "right", "timestamp", "delta", "d_delta", "d_S", "gamma_fd"]].head(6)

Rows with finite-difference gamma: 13,786


,expiration,strike,right,timestamp,delta,d_delta,d_S,gamma_fd
1054,2026-05-06,711.0,PUT,2026-05-06 10:00:00-04:00,-0.0104,0.0006,0.51,0.001176
1055,2026-05-06,711.0,PUT,2026-05-06 10:15:00-04:00,-0.0104,0.0000,0.42,0.000000
254,2026-05-06,712.0,PUT,2026-05-06 10:00:00-04:00,-0.0110,0.0003,0.51,0.000588
255,2026-05-06,712.0,PUT,2026-05-06 10:15:00-04:00,-0.0111,-0.0001,0.42,-0.000238
256,2026-05-06,712.0,PUT,2026-05-06 10:30:00-04:00,-0.0110,0.0001,-0.05,-0.002000
257,2026-05-06,712.0,PUT,2026-05-06 11:15:00-04:00,-0.0100,0.0010,1.70,0.000588


## Method 2: Black-Scholes analytical gamma

The BS gamma (identical for calls and puts) is:

$$\Gamma_{BS} = \frac{N'(d_1)}{S \cdot \sigma \cdot \sqrt{T}}$$

where $d_1 = \dfrac{\ln(S/K) + (r + \sigma^2/2)\,T}{\sigma \sqrt{T}}$

- $S$ = `underlying_price`, $K$ = `strike`, $\sigma$ = `implied_vol`  
- $T$ = `ttm_min / (252 × 390)` — time in years using trading-time convention (252 days × 390 min/day)  
- $r$ = 0.045 (approximate risk-free rate; negligible effect for 0DTE)

Rows with $T \leq 0$ or $\sigma \leq 0$ are dropped.

In [5]:
TRADING_MINUTES_PER_YEAR = 252 * 390
RISK_FREE_RATE = 0.045

df_bs = df.copy()
df_bs["T"] = df_bs["ttm_min"] / TRADING_MINUTES_PER_YEAR

valid = (df_bs["T"] > 0) & (df_bs["implied_vol"] > 0) & (df_bs["strike"] > 0)
df_bs = df_bs[valid].copy()

S = df_bs["underlying_price"].values
K = df_bs["strike"].values
sigma = df_bs["implied_vol"].values
T = df_bs["T"].values
r = RISK_FREE_RATE

d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
df_bs["gamma_bs"] = norm.pdf(d1) / (S * sigma * np.sqrt(T))

print(f"Rows with BS gamma: {len(df_bs):,}")
df_bs[["expiration", "strike", "right", "timestamp", "implied_vol", "ttm_min", "gamma_bs"]].head(6)

Rows with BS gamma: 15,652


,expiration,strike,right,timestamp,implied_vol,ttm_min,gamma_bs
0,2026-05-06,714.0,CALL,2026-05-06 10:30:00-04:00,0.3828,330.0,0.014661
1,2026-05-06,714.0,PUT,2026-05-06 09:45:00-04:00,0.3671,375.0,0.015441
2,2026-05-06,714.0,PUT,2026-05-06 10:00:00-04:00,0.3847,360.0,0.014947
3,2026-05-06,714.0,PUT,2026-05-06 10:15:00-04:00,0.4023,345.0,0.014559
4,2026-05-06,714.0,PUT,2026-05-06 10:30:00-04:00,0.3945,330.0,0.014655
5,2026-05-06,714.0,PUT,2026-05-06 10:45:00-04:00,0.4414,315.0,0.013146


## Comparison

In [6]:
# Join the two estimates on the same row identity
merge_keys = contract_keys + ["timestamp"]
both = df_fd[[*merge_keys, "gamma_fd"]].merge(
    df_bs[[*merge_keys, "gamma_bs", "ttm_min"]],
    on=merge_keys,
    how="inner",
)

# Remove obvious outliers from finite-difference noise (gamma_fd outside +-1)
both = both[both["gamma_fd"].between(-1, 1)].copy()
both["gamma_diff"] = both["gamma_bs"] - both["gamma_fd"]

corr = both[["gamma_bs", "gamma_fd"]].corr().loc["gamma_bs", "gamma_fd"]
print(f"Rows compared: {len(both):,}")
print(f"Correlation (BS vs FD):      {corr:.4f}")
print(f"Mean |gamma_bs - gamma_fd|:  {both['gamma_diff'].abs().mean():.6f}")
print(f"\ngamma_bs  --- mean={both['gamma_bs'].mean():.6f}, std={both['gamma_bs'].std():.6f}")
print(f"gamma_fd  --- mean={both['gamma_fd'].mean():.6f}, std={both['gamma_fd'].std():.6f}")

Rows compared: 13,756
Correlation (BS vs FD):      0.4096
Mean |gamma_bs - gamma_fd|:  0.050261

gamma_bs  --- mean=0.034721, std=0.030710
gamma_fd  --- mean=0.037244, std=0.104705
